In [ ]:
import sys
!"{sys.executable}" -m pip install -U pip
!"{sys.executable}" -m pip install pandas

In [ ]:
import sys
!"{sys.executable}" -m pip install matplotlib seaborn

In [ ]:
import sys
!"{sys.executable}" -m pip install scikit-learn

In [ ]:
import pandas as pd
print(pd.__version__)

In [ ]:
import pandas as pd

path = "../data/raw/2022_price1.csv" 
df = pd.read_csv(path, encoding="cp949")

print(df.shape)
df.head()

In [ ]:
for col in df.columns:
    print(col)

In [ ]:
print(df.columns.tolist())
date_col = "가격등록일자"

df[date_col] = pd.to_datetime(df[date_col].astype(str), format="%Y%m%d", errors="coerce")
print(df[date_col].min(), df[date_col].max())

In [ ]:
df_baechu = df[df["품목명"] == "배추"]

df_baechu = df_baechu[df_baechu["조사구분명"] == "도매"]

print(df_baechu.shape)
df_baechu.head()

In [ ]:
df_baechu["산물등급명"].unique()

In [ ]:
df_baechu["품목가격"].dtype

In [ ]:
df_baechu["품목가격"] = pd.to_numeric(df_baechu["품목가격"], errors="coerce")

In [ ]:
df_baechu["year"] = df_baechu["가격등록일자"].dt.year
df_baechu["month"] = df_baechu["가격등록일자"].dt.month
df_baechu["dayofweek"] = df_baechu["가격등록일자"].dt.dayofweek
df_baechu["weekofyear"] = df_baechu["가격등록일자"].dt.isocalendar().week

In [ ]:
df["시장명"].unique()

In [ ]:
df_filtered = df[
    (df["품목명"] == "배추") &
    (df["조사구분명"] == "도매")
].copy()

print(df_filtered.shape)
df_filtered["품목가격"] = pd.to_numeric(df_filtered["품목가격"], errors="coerce")
df_filtered["year"] = df_filtered["가격등록일자"].dt.year
df_filtered["month"] = df_filtered["가격등록일자"].dt.month
df_filtered["dayofweek"] = df_filtered["가격등록일자"].dt.dayofweek
df_filtered["weekofyear"] = df_filtered["가격등록일자"].dt.isocalendar().week

In [ ]:
df_filtered["품목가격"].describe()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.hist(df_filtered["품목가격"], bins=30)
plt.title("Distribution of baechu Wholesale Price")
plt.show()

In [ ]:
import numpy as np

plt.figure(figsize=(8,4))
plt.hist(np.log(df_filtered["품목가격"]), bins=30)
plt.title("Log-Transformed Price Distribution")
plt.show()

In [ ]:
import os
import pandas as pd

path = "../data/raw/"
files = sorted(os.listdir(path))

print("Files found:", files)

dfs = []

for file in files:
    if file.endswith(".csv"):
        df_temp = pd.read_csv(os.path.join(path, file), encoding="cp949")
        dfs.append(df_temp)

df_all = pd.concat(dfs, ignore_index=True)

print("Final shape:", df_all.shape)
df_all.head()

In [ ]:
df_all["가격등록일자"] = pd.to_datetime(df_all["가격등록일자"].astype(str), format="%Y%m%d")

print(df_all["가격등록일자"].min())
print(df_all["가격등록일자"].max())
print(df_all["가격등록일자"].dt.year.value_counts().sort_index())

In [ ]:
df_filtered = df_all[
    (df_all["품목명"] == "배추") &
    (df_all["조사구분명"] == "도매")
].copy()

print(df_filtered.shape)

print(df_filtered["가격등록일자"].dt.year.value_counts().sort_index())

In [ ]:
print(df_filtered["도매출하단위명"].value_counts())
print(df_filtered["도매출하단위크기"].value_counts())

In [ ]:
print(df_filtered["산물등급명"].value_counts())

In [ ]:
df_filtered = df_filtered[
    df_filtered["산물등급명"] == "상품"
].copy()

In [ ]:
print(df_filtered["할인가격여부"].value_counts())

In [ ]:
df_filtered = df_filtered[
    df_filtered["할인가격여부"] == "N"
].copy()

In [ ]:
print(df_filtered["품목가격"].describe())

In [ ]:
print("Final dataset size:", df_filtered.shape)

In [ ]:
df = df_filtered.copy()

df_train = df[df["가격등록일자"].dt.year <= 2022].copy()
df_test  = df[df["가격등록일자"].dt.year == 2023].copy()

print("train:", df_train.shape)
print("test :", df_test.shape)
print(df_train["가격등록일자"].dt.year.value_counts().sort_index())
print(df_test["가격등록일자"].dt.year.value_counts().sort_index())

In [ ]:
for d in [df_train, df_test]:
    d["month"] = d["가격등록일자"].dt.month
    d["week"] = d["가격등록일자"].dt.isocalendar().week.astype(int)
    d["dayofyear"] = d["가격등록일자"].dt.dayofyear

print(df_train[["가격등록일자","month","week","dayofyear"]].head())

In [ ]:
FEATURES = ["month", "week", "dayofyear"]
TARGET = "품목가격"

In [ ]:
df = df_filtered.copy()

df_train = df[df["가격등록일자"].dt.year <= 2022].copy()
df_test  = df[df["가격등록일자"].dt.year == 2023].copy()

print("train:", df_train.shape)
print("test :", df_test.shape)
print("train years:\n", df_train["가격등록일자"].dt.year.value_counts().sort_index())
print("test years:\n", df_test["가격등록일자"].dt.year.value_counts().sort_index())

In [ ]:
for d in [df_train, df_test]:
    d["month"] = d["가격등록일자"].dt.month
    d["week"] = d["가격등록일자"].dt.isocalendar().week.astype(int)
    d["dayofyear"] = d["가격등록일자"].dt.dayofyear

FEATURES = ["month", "week", "dayofyear"]
TARGET = "품목가격"

print(df_train[["가격등록일자"] + FEATURES].head())

In [ ]:
import numpy as np

df_train = df_train.copy()
df_test = df_test.copy()

df_train["log_price"] = np.log(df_train[TARGET])
df_test["log_price"] = np.log(df_test[TARGET])

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X_train = df_train[FEATURES]
y_train = df_train["log_price"]

X_test = df_test[FEATURES]
y_test = df_test["log_price"]

baseline = LinearRegression()
baseline.fit(X_train, y_train)

pred_log = baseline.predict(X_test)
pred_price = np.exp(pred_log)  # 다시 원 단위로 변환

In [ ]:
rmse = np.sqrt(mean_squared_error(df_test[TARGET], pred_price))
mae = mean_absolute_error(df_test[TARGET], pred_price)
r2 = r2_score(df_test[TARGET], pred_price)

print("Baseline (LinearRegression on log-price)")
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)

In [ ]:
df_test["pred_price_baseline"] = pred_price
df_test["error_baseline"] = df_test[TARGET] - df_test["pred_price_baseline"]

df_test.sort_values("error_baseline", ascending=False).head(10)[
    ["가격등록일자", "시장명", "품목명", "품종명", TARGET, "pred_price_baseline", "error_baseline"]
]

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

ridge = Ridge()

param_grid = {
    "alpha": [0.01, 0.1, 1, 10, 50, 100]
}

grid = GridSearchCV(
    ridge,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error"
)

grid.fit(X_train, y_train)

print("Best alpha:", grid.best_params_)

In [ ]:
best_ridge = grid.best_estimator_

pred_log_ridge = best_ridge.predict(X_test)
pred_price_ridge = np.exp(pred_log_ridge)

In [ ]:
rmse_ridge = np.sqrt(mean_squared_error(df_test[TARGET], pred_price_ridge))
mae_ridge = mean_absolute_error(df_test[TARGET], pred_price_ridge)
r2_ridge = r2_score(df_test[TARGET], pred_price_ridge)

print("Ridge Results")
print("RMSE:", rmse_ridge)
print("MAE :", mae_ridge)
print("R2  :", r2_ridge)

In [ ]:
for d in [df_train, df_test]:
    d["year"] = d["가격등록일자"].dt.year

In [ ]:
import numpy as np

for d in [df_train, df_test]:
    d["month_sin"] = np.sin(2 * np.pi * d["month"] / 12)
    d["month_cos"] = np.cos(2 * np.pi * d["month"] / 12)

In [ ]:
FEATURES_V2 = ["year", "week", "month_sin", "month_cos"]
print("FEATURES_V2:", FEATURES_V2)
print(df_train[FEATURES_V2].head())

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X_train2 = df_train[FEATURES_V2]
y_train = df_train["log_price"]

X_test2 = df_test[FEATURES_V2]

baseline_v2 = LinearRegression()
baseline_v2.fit(X_train2, y_train)

pred_log_v2 = baseline_v2.predict(X_test2)
pred_price_v2 = np.exp(pred_log_v2)

rmse_v2 = np.sqrt(mean_squared_error(df_test[TARGET], pred_price_v2))
mae_v2 = mean_absolute_error(df_test[TARGET], pred_price_v2)
r2_v2 = r2_score(df_test[TARGET], pred_price_v2)

print("Baseline V2 (year + seasonality sin/cos on log-price)")
print("RMSE:", rmse_v2)
print("MAE :", mae_v2)
print("R2  :", r2_v2)

In [ ]:
df_test["pred_price_baseline_v2"] = pred_price_v2
df_test["error_baseline_v2"] = df_test[TARGET] - df_test["pred_price_baseline_v2"]

In [ ]:
import numpy as np

for d in [df_train, df_test]:
    d["year"] = d["가격등록일자"].dt.year
    d["month_sin"] = np.sin(2 * np.pi * d["month"] / 12)
    d["month_cos"] = np.cos(2 * np.pi * d["month"] / 12)

FEATURES_V2 = ["year", "week", "month_sin", "month_cos"]
print("FEATURES_V2:", FEATURES_V2)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X_train2 = df_train[FEATURES_V2]
y_train = df_train["log_price"]

X_test2 = df_test[FEATURES_V2]

baseline_v2 = LinearRegression()
baseline_v2.fit(X_train2, y_train)

pred_log_base2 = baseline_v2.predict(X_test2)
pred_price_base2 = np.exp(pred_log_base2)

rmse_base2 = np.sqrt(mean_squared_error(df_test[TARGET], pred_price_base2))
mae_base2 = mean_absolute_error(df_test[TARGET], pred_price_base2)
r2_base2 = r2_score(df_test[TARGET], pred_price_base2)

print("Baseline V2")
print("RMSE:", rmse_base2)
print("MAE :", mae_base2)
print("R2  :", r2_base2)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

ridge = Ridge()

param_grid = {"alpha": [0.01, 0.1, 1, 10, 50, 100, 300, 1000]}

grid2 = GridSearchCV(
    ridge,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error"
)

grid2.fit(X_train2, y_train)

best_ridge2 = grid2.best_estimator_
print("Best alpha:", grid2.best_params_)

pred_log_ridge2 = best_ridge2.predict(X_test2)
pred_price_ridge2 = np.exp(pred_log_ridge2)

rmse_ridge2 = np.sqrt(mean_squared_error(df_test[TARGET], pred_price_ridge2))
mae_ridge2 = mean_absolute_error(df_test[TARGET], pred_price_ridge2)
r2_ridge2 = r2_score(df_test[TARGET], pred_price_ridge2)

print("Ridge V2")
print("RMSE:", rmse_ridge2)
print("MAE :", mae_ridge2)
print("R2  :", r2_ridge2)

In [ ]:
print("\n=== Comparison (2023) ===")
print(f"Baseline V1 RMSE: {rmse:.3f} | R2: {r2:.4f}")
print(f"Ridge    V1 RMSE: {rmse_ridge:.3f} | R2: {r2_ridge:.4f}")
print(f"Baseline V2 RMSE: {rmse_base2:.3f} | R2: {r2_base2:.4f}")
print(f"Ridge    V2 RMSE: {rmse_ridge2:.3f} | R2: {r2_ridge2:.4f}")

In [ ]:
df_test["pred_price_baseline_v2"] = pred_price_base2
df_test["error_baseline_v2"] = df_test[TARGET] - df_test["pred_price_baseline_v2"]

df_test["pred_price_ridge_v2"] = pred_price_ridge2
df_test["error_ridge_v2"] = df_test[TARGET] - df_test["pred_price_ridge_v2"]

In [ ]:
top_err = df_test.sort_values("error_ridge_v2", ascending=False).head(20)
top_err[["가격등록일자","시장명","시도명","시군구명","품종명",TARGET,"pred_price_ridge_v2","error_ridge_v2"]]

In [ ]:
df_test["month"] = df_test["가격등록일자"].dt.month
df_test["abs_error_ridge_v2"] = (df_test["error_ridge_v2"]).abs()

month_err = df_test.groupby("month")["abs_error_ridge_v2"].mean().sort_values(ascending=False)
print(month_err)

In [ ]:
df_test.sort_values(TARGET, ascending=False).head(10)[
    ["가격등록일자","시장명","시도명","시군구명","품종명",TARGET,"pred_price_ridge_v2","error_ridge_v2"]
]

In [ ]:
pd.get_dummies(df_train["품종명"])

In [ ]:
df_all_model = pd.concat([df_train, df_test])

df_all_model = pd.get_dummies(df_all_model, columns=["품종명"], drop_first=True)

df_train = df_all_model[df_all_model["가격등록일자"].dt.year <= 2022]
df_test  = df_all_model[df_all_model["가격등록일자"].dt.year == 2023]

FEATURES_V3 = [col for col in df_train.columns if col.startswith("품종명_")] + ["year","week","month_sin","month_cos"]

In [ ]:
X_train3 = df_train[FEATURES_V3]
X_test3 = df_test[FEATURES_V3]
y_train = df_train["log_price"]

grid3 = GridSearchCV(
    Ridge(),
    {"alpha":[0.01,0.1,1,10,50,100,300,1000]},
    cv=5,
    scoring="neg_mean_squared_error"
)

grid3.fit(X_train3, y_train)

best_ridge3 = grid3.best_estimator_

pred_log3 = best_ridge3.predict(X_test3)
pred_price3 = np.exp(pred_log3)

rmse_ridge3 = np.sqrt(mean_squared_error(df_test[TARGET], pred_price3))
r2_ridge3 = r2_score(df_test[TARGET], pred_price3)

print("Ridge V3")
print("Best alpha:", grid3.best_params_)
print("RMSE:", rmse_ridge3)
print("R2:", r2_ridge3)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

plt.plot(df_test["가격등록일자"], df_test[TARGET], label="Actual Price", alpha=0.7)
plt.plot(df_test["가격등록일자"], df_test["pred_price_ridge_v2"], label="Predicted Price (Ridge V2)", alpha=0.7)

plt.legend()
plt.title("2023 Cabbage Price: Actual vs Predicted")
plt.xlabel("Date")
plt.ylabel("Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
monthly_compare = df_test.groupby(df_test["가격등록일자"].dt.month).agg({
    TARGET: "mean",
    "pred_price_ridge_v2": "mean"
})

print(monthly_compare)

In [ ]:
df_filtered["품종명"].value_counts()

In [ ]:
df_filtered.groupby("품종명")["품목가격"].mean().sort_values(ascending=False)

In [ ]:
df_filtered.groupby("품종명")["품목가격"].std().sort_values(ascending=False)

In [ ]:
df_2023_original = df_filtered[
    df_filtered["가격등록일자"].dt.year == 2023
]

df_2023_original[df_2023_original[TARGET] > 25000]["품종명"].value_counts()

In [ ]:
varieties = df_filtered["품종명"].unique()

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

results = []

for v in varieties:
    
    df_v = df_filtered[df_filtered["품종명"] == v].copy()
    
    df_train_v = df_v[df_v["가격등록일자"].dt.year <= 2022]
    df_test_v  = df_v[df_v["가격등록일자"].dt.year == 2023]
    
    if len(df_test_v) == 0:
        continue
        
    for d in [df_train_v, df_test_v]:
        d["month"] = d["가격등록일자"].dt.month
        d["week"] = d["가격등록일자"].dt.isocalendar().week.astype(int)
        d["month_sin"] = np.sin(2 * np.pi * d["month"] / 12)
        d["month_cos"] = np.cos(2 * np.pi * d["month"] / 12)
    
    FEATURES = ["week","month_sin","month_cos"]
    
    X_train = df_train_v[FEATURES]
    y_train = np.log(df_train_v["품목가격"])
    
    X_test = df_test_v[FEATURES]
    
    grid = GridSearchCV(
        Ridge(),
        {"alpha":[0.01,0.1,1,10,50,100,300,1000]},
        cv=5,
        scoring="neg_mean_squared_error"
    )
    
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    
    pred_log = best_model.predict(X_test)
    pred_price = np.exp(pred_log)
    
    rmse = np.sqrt(mean_squared_error(df_test_v["품목가격"], pred_price))
    r2 = r2_score(df_test_v["품목가격"], pred_price)
    
    results.append((v, rmse, r2))
    
results

In [ ]:
df_sorted = df_filtered.sort_values("가격등록일자").copy()

In [ ]:
import numpy as np

df_sorted["month"] = df_sorted["가격등록일자"].dt.month
df_sorted["week"] = df_sorted["가격등록일자"].dt.isocalendar().week.astype(int)

df_sorted["month_sin"] = np.sin(2 * np.pi * df_sorted["month"] / 12)
df_sorted["month_cos"] = np.cos(2 * np.pi * df_sorted["month"] / 12)

In [ ]:
print([c for c in FEATURES_V4 if c in df_sorted.columns])
print([c for c in FEATURES_V4 if c not in df_sorted.columns])

In [ ]:
df_sorted["lag_1"] = df_sorted.groupby("품종명")["품목가격"].shift(1)
df_sorted["lag_2"] = df_sorted.groupby("품종명")["품목가격"].shift(2)
df_sorted["lag_3"] = df_sorted.groupby("품종명")["품목가격"].shift(3)

In [ ]:
df_sorted["rolling_4"] = (
    df_sorted.groupby("품종명")["품목가격"]
    .rolling(4)
    .mean()
    .reset_index(level=0, drop=True)
)

In [ ]:
df_sorted = df_sorted.dropna()

In [ ]:
FEATURES_V4 = [
    "week",
    "month_sin",
    "month_cos",
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_4"
]

In [ ]:
df_train = df_sorted[df_sorted["가격등록일자"].dt.year <= 2022].copy()
df_test  = df_sorted[df_sorted["가격등록일자"].dt.year == 2023].copy()

df_train["log_price"] = np.log(df_train["품목가격"])

X_train4 = df_train[FEATURES_V4]
y_train4 = df_train["log_price"]
X_test4  = df_test[FEATURES_V4]

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

grid4 = GridSearchCV(
    Ridge(),
    {"alpha": [0.01, 0.1, 1, 10, 50, 100, 300, 1000]},
    cv=5,
    scoring="neg_mean_squared_error"
)

grid4.fit(X_train4, y_train4)

best_ridge4 = grid4.best_estimator_
print("Best alpha:", grid4.best_params_)

pred_log4 = best_ridge4.predict(X_test4)
pred_price4 = np.exp(pred_log4)

rmse4 = np.sqrt(mean_squared_error(df_test["품목가격"], pred_price4))
r24 = r2_score(df_test["품목가격"], pred_price4)

print("Ridge V4 (lag + rolling)")
print("RMSE:", rmse4)
print("R2:", r24)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_plot = df_test[["가격등록일자", "품목가격"]].copy()
df_plot["pred_price_v4"] = pred_price4

daily = df_plot.groupby("가격등록일자").mean(numeric_only=True).reset_index()

plt.figure(figsize=(12,6))
plt.plot(daily["가격등록일자"], daily["품목가격"], label="Actual (daily avg)", alpha=0.8)
plt.plot(daily["가격등록일자"], daily["pred_price_v4"], label="Predicted V4 (daily avg)", alpha=0.8)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df_train = df_sorted[df_sorted["가격등록일자"].dt.year <= 2022].copy()

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    Ridge(),
    {"alpha":[0.01,0.1,1,10,50,100,300,1000]},
    cv=5,
    scoring="neg_mean_squared_error"
)

grid.fit(X_train, y_train)

best_alpha = grid.best_params_["alpha"]
print("Best alpha:", best_alpha)

In [ ]:
model = Ridge(alpha=best_alpha)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error

actual_2023 = (
    df_filtered[df_filtered["가격등록일자"].dt.year == 2023]
    .groupby(["품종명","가격등록일자"])["품목가격"]
    .mean()
    .reset_index()
)

merged = actual_2023.merge(pred_recursive, on=["품종명","가격등록일자"], how="inner")

rmse_by_var = (
    merged.groupby("품종명")
    .apply(lambda g: np.sqrt(mean_squared_error(g["품목가격"], g["pred_price_recursive"])))
    .sort_values()
)

print(rmse_by_var)

In [ ]:
# ============================================================
# ✅ 최종본 (교수님 구조 유지)
# - Ridge(선형+정규화) 기반
# - time(sin/cos) + lag/rolling (과거값) + season dummy
# - train: 2021-2022만 사용
# - test/predict: 2023만 예측
# - recursive forecasting (예측을 다음 lag로 피드백)
# - drift(폭주) 방지: ETA_BLEND 적용 (필수)
# - 품종별 데이터 부족 시: Global Ridge로 fallback (구조 유지)
# - 평가는 품종별 실제 price로 RMSE
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error

# -------------------------
# Settings
# -------------------------
ALPHA_GRID = np.logspace(-3, 5, 17)   # 1e-3 ~ 1e5
N_SPLITS_GLOBAL = 5
N_SPLITS_VARIETY = 3

BASE_LAGS = [1, 7, 14, 28, 52]
ROLLS = [4, 8]

USE_LOG1P = True

# ✅ 폭주 방지 핵심(봄 60000 박힘 해결용)
ETA_BLEND = 0.8   # 0.7~0.9 권장, 1.0은 drift 위험 큼

# clipping (원하면 True/False 비교)
GLOBAL_MIN_PRICE = 1000.0
GLOBAL_MAX_PRICE = 60000.0
USE_PER_VARIETY_CLIP = True
Q_LO, Q_HI = 0.01, 0.99

# 품종 모델 최소 조건(데이터 부족하면 global fallback)
MIN_TRAIN_ROWS_FOR_VARIETY_MODEL = 80
MIN_YEARS_FOR_VARIETY_MODEL = 2

# recursion seed 최소 요구치
MIN_SEED_NEEDED = 30


# -------------------------
# Helpers
# -------------------------
def season_group(month: int) -> int:
    # 0: 3-5, 1: 6-8, 2: 9-11, 3: 12-2
    if month in (3, 4, 5): return 0
    if month in (6, 7, 8): return 1
    if month in (9, 10, 11): return 2
    return 3

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["month"] = df["가격등록일자"].dt.month
    df["week"] = df["가격등록일자"].dt.isocalendar().week.astype(int)

    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)
    df["week_sin"]  = np.sin(2*np.pi*df["week"]/52)
    df["week_cos"]  = np.cos(2*np.pi*df["week"]/52)

    df["season_g"] = df["month"].apply(season_group).astype(int)
    for k in range(4):
        df[f"sg_{k}"] = (df["season_g"] == k).astype(int)
    return df

def add_lag_roll_features(df: pd.DataFrame, lags: list[int]) -> pd.DataFrame:
    df = df.copy()
    y = df["품목가격"]
    for lag in lags:
        df[f"lag_{lag}"] = y.shift(lag)
    p = y.shift(1)  # past-only anchor
    for w in ROLLS:
        df[f"roll_mean_{w}"] = p.rolling(w).mean()
    return df

def transform_y(y: pd.Series) -> np.ndarray:
    y = y.astype(float).values
    return np.log1p(y) if USE_LOG1P else np.log(y)

def inv_y(yhat: float) -> float:
    return float(np.expm1(yhat)) if USE_LOG1P else float(np.exp(yhat))

def build_features(lags: list[int], variety_list: list[str] | None) -> list[str]:
    feats = ["week_sin","week_cos","month_sin","month_cos"]
    feats += [f"lag_{lag}" for lag in lags]
    feats += [f"roll_mean_{w}" for w in ROLLS]
    feats += [f"sg_{k}" for k in range(4)]
    if variety_list is not None:
        feats += [f"v_{vv}" for vv in variety_list]
    return feats

def add_variety_dummies(df: pd.DataFrame, variety_list: list[str]) -> pd.DataFrame:
    df = df.copy()
    for vv in variety_list:
        df[f"v_{vv}"] = (df["품종명"] == vv).astype(int)
    return df

def tune_ridge_alpha(X: pd.DataFrame, y: np.ndarray, n_splits: int) -> float:
    tscv = TimeSeriesSplit(n_splits=n_splits)
    gs = GridSearchCV(
        Ridge(),
        param_grid={"alpha": ALPHA_GRID},
        scoring="neg_root_mean_squared_error",
        cv=tscv,
        n_jobs=1  # ✅ Windows UnicodeEncodeError 방지
    )
    gs.fit(X, y)
    return float(gs.best_params_["alpha"])


# ============================================================
# 0) 준비: 품종 리스트 / train(test) 분리
# ============================================================
df_filtered = df_filtered.copy()
df_filtered["가격등록일자"] = pd.to_datetime(df_filtered["가격등록일자"])
all_varieties = sorted(df_filtered["품종명"].dropna().unique().tolist())

df_train_all = df_filtered[df_filtered["가격등록일자"].dt.year <= 2022].copy()
df_test_all  = df_filtered[df_filtered["가격등록일자"].dt.year == 2023].copy()

# ============================================================
# 1) Global Ridge 학습 (fallback용) - train 2021-2022 전체
#    (구조: Ridge + time + lags + season dummy + variety dummy)
# ============================================================
global_lags = [1, 7, 14, 28]  # ✅ global은 52 빼고 안정적으로 (데이터 결측/폭주 위험 줄임)

g = add_time_features(df_train_all)
g = add_lag_roll_features(g, global_lags)
g = add_variety_dummies(g, all_varieties)

GLOBAL_FEATURES = build_features(global_lags, all_varieties)
g = g.dropna(subset=GLOBAL_FEATURES + ["품목가격"]).copy()

Xg = g[GLOBAL_FEATURES].astype(float)
yg = transform_y(g["품목가격"])

global_alpha = tune_ridge_alpha(Xg, yg, N_SPLITS_GLOBAL)
global_model = Ridge(alpha=global_alpha).fit(Xg, yg)

print("Global alpha:", global_alpha, "Global train rows:", len(g))


# ============================================================
# 2) 품종별 모델 학습(가능한 품종만) + recursive 예측
#    - 품종 데이터 부족이면 global fallback
#    - 폭주 방지: ETA_BLEND로 lag 피드백 안정화
# ============================================================
pred_rows = []
model_catalog = {}  # v -> dict or None (fallback)

for v in all_varieties:
    df_v = df_filtered[df_filtered["품종명"] == v].sort_values("가격등록일자").copy()
    train_raw = df_v[df_v["가격등록일자"].dt.year <= 2022].copy()
    test_raw  = df_v[df_v["가격등록일자"].dt.year == 2023].copy()

    if len(test_raw) == 0:
        continue

    # -------- seed 가능 여부(최소 past 필요)
    seed_series = train_raw["품목가격"].dropna().values
    if len(seed_series) < MIN_SEED_NEEDED:
        # 너무 부족하면 예측 자체를 못함(이 경우는 거의 없음)
        continue

    # -------- 품종별 가능한 lag 자동 축소 (데이터 부족하면 52 제거 등)
    n_avail = len(seed_series)
    max_ok = n_avail - 5  # 안전 마진
    lags_v = [lag for lag in BASE_LAGS if lag <= max_ok]
    if len(lags_v) == 0:
        lags_v = [1]

    # -------- 품종별 clip 범위(옵션)
    if USE_PER_VARIETY_CLIP:
        # train 분포 기반 (너가 원래 하던 방식 유지)
        # train_raw에 있는 값으로 계산
        v_min = float(train_raw["품목가격"].quantile(Q_LO))
        v_max = float(train_raw["품목가격"].quantile(Q_HI))
        v_min = max(v_min, GLOBAL_MIN_PRICE)
        v_max = min(v_max, GLOBAL_MAX_PRICE)
        if v_min >= v_max:
            v_min, v_max = GLOBAL_MIN_PRICE, GLOBAL_MAX_PRICE
    else:
        v_min, v_max = GLOBAL_MIN_PRICE, GLOBAL_MAX_PRICE

    # -------- 품종 모델 학습 가능 조건 체크
    tv = add_time_features(train_raw)
    tv = add_lag_roll_features(tv, lags_v)

    FEATURES_V = build_features(lags_v, variety_list=None)
    tv = tv.dropna(subset=FEATURES_V + ["품목가격"]).copy()

    years_in_train = train_raw["가격등록일자"].dt.year.nunique()
    use_variety_model = (len(tv) >= MIN_TRAIN_ROWS_FOR_VARIETY_MODEL) and (years_in_train >= MIN_YEARS_FOR_VARIETY_MODEL)

    if use_variety_model:
        Xv = tv[FEATURES_V].astype(float)
        yv = transform_y(tv["품목가격"])
        a = tune_ridge_alpha(Xv, yv, N_SPLITS_VARIETY)
        m = Ridge(alpha=a).fit(Xv, yv)
        model_catalog[v] = {"model": m, "alpha": a, "lags": lags_v, "features": FEATURES_V, "mode": "variety"}
    else:
        model_catalog[v] = None  # global fallback

    # -------- recursive seed (raw train 마지막 값들, dropna 후가 아니라 raw에서!)
    need = max(max(lags_v), max(ROLLS))
    if len(seed_series) < need:
        # lag가 길어져서 seed 부족하면 줄여서라도 예측
        lags_v = [lag for lag in lags_v if lag <= len(seed_series)]
        if len(lags_v) == 0:
            lags_v = [1]
        need = max(max(lags_v), max(ROLLS))
        if len(seed_series) < need:
            continue

    lag_vals = list(seed_series[-need:])

    test_dates = (
        test_raw[["가격등록일자"]]
        .drop_duplicates()
        .sort_values("가격등록일자")["가격등록일자"]
        .tolist()
    )

    for dt in test_dates:
        month = dt.month
        week  = dt.isocalendar().week

        # 공통 row (time + season dummy)
        row = {
            "month_sin": np.sin(2*np.pi*month/12),
            "month_cos": np.cos(2*np.pi*month/12),
            "week_sin":  np.sin(2*np.pi*week/52),
            "week_cos":  np.cos(2*np.pi*week/52),
        }
        sg = season_group(month)
        for k in range(4):
            row[f"sg_{k}"] = 1 if sg == k else 0

        # lags + rolls (past-only)
        for lag in lags_v:
            row[f"lag_{lag}"] = float(lag_vals[-lag])
        for w in ROLLS:
            row[f"roll_mean_{w}"] = float(np.mean(lag_vals[-w:]))

        # -------- 모델 선택: 품종 모델 있으면 사용, 없으면 global
        if model_catalog[v] is not None:
            info = model_catalog[v]
            feats = info["features"]
            x_df = pd.DataFrame([row], columns=feats).astype(float)
            pred_t = float(info["model"].predict(x_df)[0])
            used_mode = "variety"
            used_alpha = float(info["alpha"])
        else:
            # global은 품종 더미가 필요
            row_g = dict(row)
            for vv in all_varieties:
                row_g[f"v_{vv}"] = 1 if vv == v else 0
            x_df = pd.DataFrame([row_g], columns=GLOBAL_FEATURES).astype(float)
            pred_t = float(global_model.predict(x_df)[0])
            used_mode = "global"
            used_alpha = float(global_alpha)

        pred_price = inv_y(pred_t)

        # ✅ 안정화: 클리핑 (global + per-variety)
        pred_price = float(np.clip(pred_price, GLOBAL_MIN_PRICE, GLOBAL_MAX_PRICE))
        pred_price = float(np.clip(pred_price, v_min, v_max))

        pred_rows.append({
            "품종명": v,
            "가격등록일자": dt,
            "pred_price_recursive": pred_price,
            "model_used": used_mode,
            "alpha_used": used_alpha
        })

        # ✅ drift 방지: 블렌딩으로 lag 업데이트
        blended = ETA_BLEND * pred_price + (1.0 - ETA_BLEND) * lag_vals[-1]
        lag_vals.append(float(blended))

pred_recursive = pd.DataFrame(pred_rows)
print("pred_recursive rows:", len(pred_recursive))


# ============================================================
# 3) Evaluate (품종별 실제값 기준)
# ============================================================
actual_2023 = (
    df_filtered[df_filtered["가격등록일자"].dt.year == 2023]
    .groupby(["품종명","가격등록일자"])["품목가격"]
    .mean()
    .reset_index()
)

merged = actual_2023.merge(pred_recursive, on=["품종명","가격등록일자"], how="inner")
print("merged rows:", len(merged))

rmse_by_var = (
    merged.groupby("품종명")
    .apply(lambda g: np.sqrt(mean_squared_error(g["품목가격"], g["pred_price_recursive"])))
    .sort_values()
)

print("\nRMSE by variety:")
print(rmse_by_var)

print("\nModel usage by variety:")
print(merged.groupby(["품종명","model_used"]).size().unstack(fill_value=0))


# ============================================================
# 4) Plot helper
# ============================================================
def plot_variety(var_name: str):
    g = merged[merged["품종명"] == var_name].sort_values("가격등록일자")
    print(var_name, "rows:", len(g))
    if len(g) == 0:
        return
    plt.figure(figsize=(12,6))
    plt.plot(g["가격등록일자"], g["품목가격"], label="Actual")
    plt.plot(g["가격등록일자"], g["pred_price_recursive"], label="Pred (recursive)")
    plt.title(f"2023 Actual vs Pred - {var_name}")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 예시
plot_variety("여름(고랭지)")
plot_variety("봄")
plot_variety("월동")
plot_variety("가을")

In [ ]:
# ============================================================
# ✅ 완성본 (교수님 구조 유지)
# - 필터: 여름(고랭지) 제외
# - 모델: 품종별 Ridge (GridSearch + TimeSeriesSplit)
# - feature: time(sin/cos) + past-only lag/rolling
# - train: 2021-2022 / test&predict: 2023
# - 예측 방식:
#    * 월동/가을: recursive + ETA_BLEND (0.8)
#    * 봄: soft-recursive + SPRING_BLEND (0.4)  ← 직선 문제 해결 + 폭주 방지
# - 안정화: 전 단계에서 clip 적용(입력/출력 모두)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error

# -------------------------
# Plot font (Windows 한글 경고 제거)
# -------------------------
mpl.rcParams["font.family"] = "Malgun Gothic"
mpl.rcParams["axes.unicode_minus"] = False

# -------------------------
# Settings
# -------------------------
EXCLUDE_VARIETY = "여름(고랭지)"

ALPHA_GRID = np.logspace(-3, 5, 17)
N_SPLITS = 3

BASE_LAGS = [1, 7, 14, 28]
ROLLS = [4, 8]

USE_LOG1P = True

GLOBAL_MIN_PRICE = 1000.0
GLOBAL_MAX_PRICE = 60000.0

# recursive 안정화
ETA_BLEND = 0.8          # 월동/가을용
SPRING_BLEND = 0.4       # 봄용 (soft-recursive)

# -------------------------
# Helpers
# -------------------------
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["month"] = df["가격등록일자"].dt.month
    df["week"] = df["가격등록일자"].dt.isocalendar().week.astype(int)

    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)
    df["week_sin"]  = np.sin(2*np.pi*df["week"]/52)
    df["week_cos"]  = np.cos(2*np.pi*df["week"]/52)

    y = df["품목가격"]
    for lag in BASE_LAGS:
        df[f"lag_{lag}"] = y.shift(lag)

    p = y.shift(1)  # past-only
    for w in ROLLS:
        df[f"roll_mean_{w}"] = p.rolling(w).mean()

    return df

def transform_y(y: pd.Series) -> np.ndarray:
    y = y.astype(float).values
    return np.log1p(y) if USE_LOG1P else np.log(y)

def inv_y(yhat: float) -> float:
    return float(np.expm1(yhat)) if USE_LOG1P else float(np.exp(yhat))

FEATURES = (
    ["week_sin","week_cos","month_sin","month_cos"] +
    [f"lag_{l}" for l in BASE_LAGS] +
    [f"roll_mean_{w}" for w in ROLLS]
)

def tune_alpha(X: pd.DataFrame, y: np.ndarray) -> float:
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    gs = GridSearchCV(
        Ridge(),
        {"alpha": ALPHA_GRID},
        scoring="neg_root_mean_squared_error",
        cv=tscv,
        n_jobs=1  # Windows Unicode error 방지
    )
    gs.fit(X, y)
    return float(gs.best_params_["alpha"])

def safe_clip(x: float) -> float:
    return float(np.clip(x, GLOBAL_MIN_PRICE, GLOBAL_MAX_PRICE))

# ============================================================
# Data (여름 제외)
# ============================================================
df_model = df_filtered[df_filtered["품종명"] != EXCLUDE_VARIETY].copy()
df_model["가격등록일자"] = pd.to_datetime(df_model["가격등록일자"])

df_train = df_model[df_model["가격등록일자"].dt.year <= 2022].copy()
df_test  = df_model[df_model["가격등록일자"].dt.year == 2023].copy()

# ============================================================
# Fit per variety + Predict 2023
# ============================================================
pred_rows = []
alpha_rows = []

for v in sorted(df_model["품종명"].dropna().unique()):
    train_raw = df_train[df_train["품종명"] == v].sort_values("가격등록일자").copy()
    test_raw  = df_test[df_test["품종명"] == v].sort_values("가격등록일자").copy()

    if len(test_raw) == 0 or len(train_raw) < 50:
        continue

    train_v = add_features(train_raw)
    train_v = train_v.dropna(subset=FEATURES + ["품목가격"]).copy()
    if len(train_v) < 50:
        continue

    X = train_v[FEATURES].astype(float)
    y = transform_y(train_v["품목가격"])

    alpha_best = tune_alpha(X, y)
    model = Ridge(alpha=alpha_best).fit(X, y)
    alpha_rows.append({"품종명": v, "alpha": alpha_best, "n_train": len(train_v)})

    # seed for recursion
    need_seed = max(max(BASE_LAGS), max(ROLLS))
    seed = train_raw["품목가격"].dropna().values
    if len(seed) < need_seed:
        continue
    seed = np.clip(seed, GLOBAL_MIN_PRICE, GLOBAL_MAX_PRICE)
    lag_vals = list(seed[-need_seed:])

    test_dates = (
        test_raw[["가격등록일자"]]
        .drop_duplicates()
        .sort_values("가격등록일자")["가격등록일자"]
        .tolist()
    )

    for dt in test_dates:
        month = dt.month
        week = dt.isocalendar().week

        row = {
            "month_sin": np.sin(2*np.pi*month/12),
            "month_cos": np.cos(2*np.pi*month/12),
            "week_sin":  np.sin(2*np.pi*week/52),
            "week_cos":  np.cos(2*np.pi*week/52),
        }

        # lag/rolling (항상 안전클립)
        for lag in BASE_LAGS:
            row[f"lag_{lag}"] = safe_clip(lag_vals[-lag])

        for w in ROLLS:
            row[f"roll_mean_{w}"] = safe_clip(np.mean(lag_vals[-w:]))

        x_df = pd.DataFrame([row], columns=FEATURES).astype(float)

        pred_t = float(model.predict(x_df)[0])
        pred_price = safe_clip(inv_y(pred_t))

        pred_rows.append({
            "품종명": v,
            "가격등록일자": dt,
            "pred_price": pred_price
        })

        # ✅ 품종별 피드백 강도
        blend = SPRING_BLEND if v == "봄" else ETA_BLEND
        blended = safe_clip(blend * pred_price + (1.0 - blend) * safe_clip(lag_vals[-1]))
        lag_vals.append(blended)

pred_df = pd.DataFrame(pred_rows)
alpha_df = pd.DataFrame(alpha_rows).sort_values("alpha")

print("pred_df rows:", len(pred_df))
display(alpha_df)

# ============================================================
# Evaluate (품종별 실제값 기준)
# ============================================================
actual_2023 = (
    df_model[df_model["가격등록일자"].dt.year == 2023]
    .groupby(["품종명","가격등록일자"])["품목가격"]
    .mean()
    .reset_index()
)

merged = actual_2023.merge(pred_df, on=["품종명","가격등록일자"], how="inner")
print("merged rows:", len(merged))

rmse_by_var = (
    merged.groupby("품종명")
    .apply(lambda g: np.sqrt(mean_squared_error(g["품목가격"], g["pred_price"])))
    .sort_values()
)

print("\nRMSE by variety (여름 제외, 봄 soft-recursive):")
print(rmse_by_var)

# ============================================================
# Plot helper
# ============================================================
def plot_variety(var_name: str):
    g = merged[merged["품종명"] == var_name].sort_values("가격등록일자")
    print(var_name, "rows:", len(g))
    if len(g) == 0:
        return
    plt.figure(figsize=(12,6))
    plt.plot(g["가격등록일자"], g["품목가격"], label="Actual")
    plt.plot(g["가격등록일자"], g["pred_price"], label="Pred")
    plt.title(f"2023 Actual vs Pred - {var_name}")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_variety("봄")
plot_variety("월동")
plot_variety("가을")

In [ ]:
# ============================================================
# 🔬 Research Extension Model (Resume-Level Implementation)
# - Multicollinearity analysis
# - Scaling
# - PCA
# - Ridge vs Lasso
# - 5-fold CV
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

# -------------------------
# Use train data only (2021-2022)
# -------------------------
df_research = df_train.copy()
df_research = add_features(df_research)
df_research = df_research.dropna(subset=FEATURES + ["품목가격"]).copy()

X = df_research[FEATURES].astype(float)
y = np.log1p(df_research["품목가격"].values)

print("Research dataset shape:", X.shape)

# ============================================================
# 1️⃣ Multicollinearity Analysis
# ============================================================

corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_features = [column for column in upper.columns if any(upper[column] > 0.85)]

print("Highly correlated features (|corr| > 0.85):")
print(high_corr_features)

# Drop high correlation features
X_reduced = X.drop(columns=high_corr_features)
print("After dropping high-corr features:", X_reduced.shape)

# ============================================================
# 2️⃣ Ridge vs Lasso without PCA
# ============================================================

tscv = TimeSeriesSplit(n_splits=5)

ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(max_iter=5000))
])

ridge_grid = {"model__alpha": np.logspace(-3, 3, 13)}
lasso_grid = {"model__alpha": np.logspace(-3, 1, 9)}

ridge_gs = GridSearchCV(ridge_pipe, ridge_grid, cv=tscv,
                        scoring="neg_root_mean_squared_error", n_jobs=1)
lasso_gs = GridSearchCV(lasso_pipe, lasso_grid, cv=tscv,
                        scoring="neg_root_mean_squared_error", n_jobs=1)

ridge_gs.fit(X_reduced, y)
lasso_gs.fit(X_reduced, y)

ridge_rmse = -ridge_gs.best_score_
lasso_rmse = -lasso_gs.best_score_

print("\nRidge RMSE:", ridge_rmse)
print("Lasso RMSE:", lasso_rmse)

# ============================================================
# 3️⃣ PCA + Ridge
# ============================================================

pca_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("model", Ridge())
])

pca_grid = {
    "pca__n_components": list(range(3, min(15, X_reduced.shape[1]))),
    "model__alpha": np.logspace(-3, 3, 13)
}

pca_gs = GridSearchCV(pca_pipe, pca_grid, cv=tscv,
                      scoring="neg_root_mean_squared_error", n_jobs=1)

pca_gs.fit(X_reduced, y)

pca_rmse = -pca_gs.best_score_

print("\nPCA + Ridge RMSE:", pca_rmse)
print("Best PCA components:", pca_gs.best_params_["pca__n_components"])

# Explained variance
best_pca = pca_gs.best_estimator_.named_steps["pca"]
print("\nExplained variance ratio (cumulative):")
print(np.cumsum(best_pca.explained_variance_ratio_))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

# -------------------------
# 준비: 2021-2022 연구용 데이터
# -------------------------
df_research = df_filtered[df_filtered["가격등록일자"].dt.year <= 2022].copy()
df_research = add_features(df_research)
df_research = df_research.dropna(subset=FEATURES + ["품목가격"]).copy()

X0 = df_research[FEATURES].astype(float)
y  = np.log1p(df_research["품목가격"].astype(float).values)

print("Rows:", len(df_research), "Original features:", X0.shape[1])

def drop_high_corr(X, thresh):
    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_cols = [c for c in upper.columns if any(upper[c] > thresh)]
    return X.drop(columns=drop_cols), drop_cols

tscv = TimeSeriesSplit(n_splits=5)

def run_ridge_lasso(X, tag):
    ridge_pipe = Pipeline([("scaler", StandardScaler()), ("model", Ridge())])
    lasso_pipe = Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=20000))])

    ridge_grid = {"model__alpha": np.logspace(-3, 3, 13)}
    lasso_grid = {"model__alpha": np.logspace(-4, 1, 12)}

    rg = GridSearchCV(ridge_pipe, ridge_grid, cv=tscv,
                      scoring="neg_root_mean_squared_error", n_jobs=1)
    lg = GridSearchCV(lasso_pipe, lasso_grid, cv=tscv,
                      scoring="neg_root_mean_squared_error", n_jobs=1)

    rg.fit(X, y)
    lg.fit(X, y)

    return {
        "setting": tag,
        "n_features": X.shape[1],
        "ridge_rmse": -rg.best_score_,
        "ridge_alpha": rg.best_params_["model__alpha"],
        "lasso_rmse": -lg.best_score_,
        "lasso_alpha": lg.best_params_["model__alpha"],
    }

def run_pca_ridge(X, tag):
    n_feat = X.shape[1]
    comps = list(range(2, min(10, n_feat) + 1)) if n_feat >= 2 else [1]

    pipe = Pipeline([("scaler", StandardScaler()),
                     ("pca", PCA()),
                     ("model", Ridge())])

    grid = {
        "pca__n_components": comps,
        "model__alpha": np.logspace(-3, 3, 13)
    }

    gs = GridSearchCV(pipe, grid, cv=tscv,
                      scoring="neg_root_mean_squared_error", n_jobs=1)
    gs.fit(X, y)

    best_est = gs.best_estimator_
    best_pca = best_est.named_steps["pca"]
    cum_var  = np.cumsum(best_pca.explained_variance_ratio_)

    return {
        "setting": tag,
        "n_features": X.shape[1],
        "pca_ridge_rmse": -gs.best_score_,
        "best_k": gs.best_params_["pca__n_components"],
        "best_alpha": gs.best_params_["model__alpha"],
        "cum_explained_var_last": float(cum_var[-1])
    }

results = []

# (1) corr 0.85 drop
X85, drop85 = drop_high_corr(X0, 0.85)
print("\n[corr>0.85 drop] remaining:", X85.shape[1], "dropped:", len(drop85))
results.append(run_ridge_lasso(X85, "corr_drop_0.85"))

# (2) corr 0.95 drop (PCA도 의미있게)
X95, drop95 = drop_high_corr(X0, 0.95)
print("\n[corr>0.95 drop] remaining:", X95.shape[1], "dropped:", len(drop95))
results.append(run_ridge_lasso(X95, "corr_drop_0.95"))
results.append(run_pca_ridge(X95, "PCA+Ridge on corr_drop_0.95"))

# (3) no drop PCA
results.append(run_pca_ridge(X0, "PCA+Ridge on no_drop"))

res_df = pd.DataFrame(results)
display(res_df)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge

# 1) X0, y0 다시 만들기 (2021-2022 research set 기준으로 "행 정렬" 보장)
df_research = df_filtered[df_filtered["가격등록일자"].dt.year <= 2022].copy()
df_research = add_features(df_research)
df_research = df_research.dropna(subset=FEATURES + ["품목가격"]).copy()

X0 = df_research[FEATURES].astype(float)
y0 = np.log1p(df_research["품목가격"].astype(float))  # pandas Series (index 유지)

# 2) X_reduced를 만들었던 corr-drop(0.95) 동일하게 다시 적용 (index 유지)
def drop_high_corr(X_in: pd.DataFrame, thresh: float):
    corr = X_in.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_cols = [c for c in upper.columns if any(upper[c] > thresh)]
    return X_in.drop(columns=drop_cols), drop_cols

X_reduced_095, dropped = drop_high_corr(X0, 0.95)

# 3) 혹시라도 NaN/inf 생겼으면 제거 + y도 같은 index로 맞춤
X_reduced_095 = X_reduced_095.replace([np.inf, -np.inf], np.nan).dropna()
y_aligned = y0.loc[X_reduced_095.index].astype(float).values

print("X rows:", X_reduced_095.shape[0], "y rows:", len(y_aligned))
print("Dropped features:", len(dropped), dropped)

# 4) Baseline vs Ridge variance 비교 (Pipeline로 scaler 포함)
tscv = TimeSeriesSplit(n_splits=5)

baseline_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

ridge_best_alpha = ridge_gs.best_params_["model__alpha"] if "ridge_gs" in globals() else 1000.0
ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=float(ridge_best_alpha)))
])

baseline_scores = -cross_val_score(
    baseline_pipe, X_reduced_095, y_aligned,
    cv=tscv, scoring="neg_root_mean_squared_error"
)

ridge_scores = -cross_val_score(
    ridge_pipe, X_reduced_095, y_aligned,
    cv=tscv, scoring="neg_root_mean_squared_error"
)

print("\nBaseline fold RMSE:", baseline_scores)
print("Baseline mean:", baseline_scores.mean(), "var:", baseline_scores.var())

print("\nRidge fold RMSE:", ridge_scores)
print("Ridge mean:", ridge_scores.mean(), "var:", ridge_scores.var())

var_red = (baseline_scores.var() - ridge_scores.var()) / baseline_scores.var() * 100
print("\nVariance reduction (%):", var_red)